## Text Extraction, Cleaning, and Rephrasing from multiple books

In [1]:
import pandas as pd
import traceback
import json
from PyPDF2 import PdfReader
import os

from bson import ObjectId
from motor.motor_asyncio import AsyncIOMotorClient
from langchain_openai import ChatOpenAI
from langchain import PromptTemplate, LLMChain

from case_study_prompts import (
EXTRACTION_SYSTEM,
EXTRACTION_USER,
EXTRACTION_OUTPUT_FORMAT,
EMPTY_EXTRACTION_OUTPUT_FORMAT,
CLEAN_REPHRASE_USER,
CLEAN_REPHRASE_SYSTEM,
REPHRASE_OUTPUT_FORMAT,
REPHRASE_OUTPUT_EXAMPLE,
BIOLOGY_SOLUTION_SYSTEM_PROMPT,
CHEMISTRY_SOLUTION_SYSTEM_PROMPT,
MATHEMATICS_SOLUTION_SYSTEM_PROMPT,
PHYSICS_SOLUTION_SYSTEM_PROMPT,
DISTRACTOR_SYSTEM_PROMPT
)

from config import (
    OPENAI_API_KEY,
    MONGO_URL,
    DB_NAME
)

In [2]:
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "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"
# os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
# os.environ["LANGCHAIN_PROJECT"] = "Questions Extractor"

# Pipeline

In [2]:
import json
def parse_Response(response):
    if isinstance(response, str):
        # print(response)
        start_index = response.find("{")
        end_index = response.rfind("}")
        if start_index != -1 and end_index != -1:
            valid_json_content = response[start_index : end_index + 1]
            try:
                JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
                # append_list_to_file(JSON_response)
                return JSON_response
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON response: {e.__class__.__name__} - {e}\n\n Still trying to work on particular exceptions ...")
                if str(e).startswith("Extra data"):
                    json_parts = valid_json_content.split('}\n{')
                    if json_parts:
                        first_json = json_parts[0] + '}'
                        return parse_Response(first_json)  
                elif str(e).startswith("Invalid \escape"):
                    print("Before removing \\ issue: ", valid_json_content)
                    str1 = valid_json_content.replace("\\\\\\\\\\", "fvback")
                    str1 = str1.replace("\\\\\\\\", "frback")
                    str1 = str1.replace("\\\\\\", "trlback")
                    str1 = str1.replace("\\\\", "dblback")
                    str1 = str1.replace("\\", "\\\\")
                    str1 = str1.replace("fvback","\\\\\\\\\\")
                    str1 = str1.replace("frback", "\\\\\\\\")
                    str1 = str1.replace("trlback","\\\\\\")
                    strfinal = str1.replace("dblback","\\\\")
                    print("After removing \\ issue: ", strfinal)
                    JSON_response = json.loads(strfinal.replace("\n", ""))
                    return JSON_response
                else:
                    print("Actual Content: ", valid_json_content)
        else:
            print("No valid JSON content found in the response.")
        # time.sleep(50)
    elif isinstance(response, dict):
        return response
    else:
        print("No response message found", type(response))

<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_25956\3339088456.py:10: SyntaxWarning: invalid escape sequence '\('
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_25956\3339088456.py:10: SyntaxWarning: invalid escape sequence '\)'
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_25956\3339088456.py:20: SyntaxWarning: invalid escape sequence '\e'
  elif str(e).startswith("Invalid \escape"):


In [3]:
# All question collection

def collect_questions_from_chapter_with_Langchain(subject, chapter_text, chapter_name, grade):
    try:
        # initialize the ChapOpenAI object
        llm = ChatOpenAI(
            model_name="gpt-4o",
            api_key= OPENAI_API_KEY,
            temperature=0.2,
        )
        # match subject:
        #     case "mathematics":
        #         examples=MATHEMATICS_EXTRACTION_OUTPUT_EXAMPLE
        #     case "biology":
        #         examples=BIOLOGY_EXTRACTION_OUTPUT_EXAMPLE
        #     case "physics":
        #         examples =PHYSICS_EXTRACTION_OUTPUT_EXAMPLE
        #     case "chemistry":
        #         examples=CHEMISTRY_EXTRACTION_OUTPUT_EXAMPLE

        #formatting prompts
        system = EXTRACTION_SYSTEM.format("",grade = grade, subject = subject,EMPTY_EXTRACTION_OUTPUT_FORMAT = EMPTY_EXTRACTION_OUTPUT_FORMAT, out_format = EXTRACTION_OUTPUT_FORMAT)
        user = EXTRACTION_USER.format(chapter_name = chapter_name, chapter_text = chapter_text)

        #generating response from api call
        response = llm.invoke(
            [
                ("system", system),
                ("human", user)
            ],
        )
        return response
    except Exception as e:
        print("Error with in extraction: ", type(e).__name__, "–", e, "\n", traceback.format_exc())
        return None

In [4]:
def clean_rephrase_question(question_text, topic_list):
    try:
        # initialize the ChapOpenAI object
        llm = ChatOpenAI(
            model_name="gpt-4o",
            api_key= OPENAI_API_KEY,
            temperature=0.2,
            # verbose = True
        )

        #formatting prompts
        system = CLEAN_REPHRASE_SYSTEM.format("",out_example = REPHRASE_OUTPUT_EXAMPLE, out_format = REPHRASE_OUTPUT_FORMAT)
        user = CLEAN_REPHRASE_USER.format(question_text = question_text, topic_list = topic_list)

        #generating response from api call
        response = llm.invoke(
            [
                ("system", system),
                ("human", user)
            ]
        )
        return response
    except Exception as e:
        print("Error cleaning text: ", type(e).__name__, "–", e, "\n", traceback.format_exc())
        return question_text  # Return original text in case of an error

In [5]:
import os
import traceback
from pathlib import Path
import zipfile
import tempfile
import shutil

def read_data_from_latex(publication: str, chapter_name: str) -> str:
    """
    Read LaTeX content from a specified publication and chapter zip file.
    
    Args:
        publication (str): Name of the publication (e.g., 'mtg')
        chapter_name (str): Name of the chapter (e.g., 'Life Processes')
    
    Returns:
        str: Content of the LaTeX file if successful, empty string if failed
    """
    try:
        # Construct the zip file path
        zip_path = os.path.join(r"textbooks", publication, f"{chapter_name}.zip")
        
        # Verify zip file exists
        if not os.path.exists(zip_path):
            raise FileNotFoundError(f"Zip file not found: {zip_path}")
        
        # Create a temporary directory to extract files
        with tempfile.TemporaryDirectory() as temp_dir:
            # Extract the zip file
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(temp_dir)
            
            # Find directories starting with 2025
            content_dirs = [d for d in os.listdir(temp_dir) 
                          if os.path.isdir(os.path.join(temp_dir, d)) and 
                          d.startswith('2025')]
            
            if not content_dirs:
                raise FileNotFoundError(f"No content directories found in zip file")
                
            # Get the latest directory
            latest_dir = sorted(content_dirs)[-1]
            dir_path = os.path.join(temp_dir, latest_dir)
            
            # Find the .tex file in the directory
            tex_files = [f for f in os.listdir(dir_path) if f.endswith('.tex')]
                    
            if not tex_files:
                raise FileNotFoundError(f"No .tex file found in {dir_path}")
                
            # Full path to the tex file
            latex_path = os.path.join(dir_path, tex_files[0])
            
            # Read the content
            with open(latex_path, 'r', encoding='utf-8') as file:
                content = file.read()
                
            print(f"Successfully processed LaTeX file from zip: {latex_path}")
            return content

    except Exception as e:
        print(
            "Failed to process the LaTeX file. Exception Occurred:",
            type(e).__name__,
            "–",
            e,
            "\n",
            traceback.format_exc()
        )
        return ""

In [25]:
def extract_questions(subject: str, chapter_texts: list, chapter_name: str, grade: str) -> dict:
    try:
        if not isinstance(chapter_texts, list) or not chapter_texts:
            return {}
            
        formatted_output = {}
        chunk_size = 500
        case_study_counter = 1
        
        for i in range(0, len(chapter_texts), chunk_size):
            chunk = chapter_texts[i:i + chunk_size]
            if not chunk:
                continue
                
            print(f"\nProcessing chunk {i//chunk_size + 1}/{-(-len(chapter_texts)//chunk_size)}")
            
            response = collect_questions_from_chapter_with_Langchain(
                subject=subject,
                chapter_text='\n'.join(chunk),
                chapter_name=chapter_name,
                grade=grade
            )
            
            if response and response.content:
                print("\nRaw response content:", response.content[:200])  # Print first 200 chars
                parsed = parse_Response(response.content)
                print("\nParsed response:", parsed)  # Print parsed response
                
                if parsed:
                    # Check if parsed response has case studies directly
                    if isinstance(parsed.get("case_studies"), dict):
                        case_studies = parsed["case_studies"]
                    else:
                        case_studies = parsed  # Assume the whole response is case studies
                        
                    for content in case_studies.values():
                        if isinstance(content, dict):
                            if "description" not in content or "questions" not in content:
                                print(f"\nMissing required fields. Content structure: {content.keys()}")
                                continue
                                
                            new_key = f"case_study_{case_study_counter}"
                            formatted_output[new_key] = {
                                "description": content["description"],
                                "questions": content["questions"]
                            }
                            print(f"\nExtracted questions from {new_key}")
                            print(f"Number of questions: {len(content['questions'])}")
                            case_study_counter += 1
            
            print(f"\nCompleted processing chunk {i//chunk_size + 1}")
                
        total_questions = sum(len(case_study["questions"]) for case_study in formatted_output.values())
        print(f"\nTotal questions extracted: {total_questions}")
        return formatted_output

    except Exception as e:
        print(f"Error: {str(e)}")
        return {}

In [26]:
def rephrase_questions(conf_data: dict, extracted_questions: dict, topics: list) -> dict:
   print("Starting the cleanup and rephrasing process...")
   try:
       if not conf_data or "chapter_name" not in conf_data:
           raise ValueError("Missing chapter_name in configuration")
           
       if not topics:
           raise ValueError("No topics found in the data")
           
       if not extracted_questions:
           raise ValueError(f"No questions found")
       
       formatted_output = {}
       
       # Process one case study at a time
       for case_study_key, content in extracted_questions.items():
           try:
               print(f"\nProcessing {case_study_key}")
               
               # Clean and rephrase the case study
               cleaned_response = clean_rephrase_question(content, topics)
               
               if cleaned_response and cleaned_response.content:
                   parsed = parse_Response(cleaned_response.content)
                   if parsed and isinstance(parsed, dict):
                       # Keep the same case_study_key from input
                       for key, case_study in parsed.items():
                           if isinstance(case_study, dict) and all(k in case_study for k in ["description", "questions", "topic", "topic_id"]):
                               formatted_output[case_study_key] = case_study
                               print(f"Successfully processed {case_study_key}")
                               print(f"Mapped topic: {case_study['topic']}")
                   else:
                       print(f"Warning: Invalid format in cleaned response for {case_study_key}")
               
               print("-" * 50)
               
           except Exception as e:
               print(f"Error processing {case_study_key}: {str(e)}")
               continue
       
       total_questions = sum(len(case_study["questions"]) for case_study in formatted_output.values())
       print(f"Total questions processed: {total_questions}")
       return formatted_output

   except Exception as e:
       print(f"Error: {str(e)}")
       return {}

In [20]:
# get chapter and book data from defaultConf
with open("defaultConf.json", "r") as f:
    conf_data = json.load(f)

chapter = conf_data["chapter_name"]
subject = conf_data["subject"]


print("Chapter to work on: ", chapter)
print(subject)
print(conf_data['publication'])

Chapter to work on:  Chemical Reactions and Equations
chemistry
mtg


In [21]:
from typing import Tuple, Dict, Optional
import traceback
import logging

def extract_rephrase_questions(conf_data: dict, topics: list) -> Tuple[Optional[Dict], Optional[Dict]]:
    try:
        # Validate configuration data
        required_fields = ["subject", "grade", "chapter_name"]
        missing_fields = [field for field in required_fields if field not in conf_data]
        if missing_fields:
            raise ValueError(f"Missing required configuration fields: {', '.join(missing_fields)}")
            
        if not topics:
            raise ValueError("No topics provided for question rephrasing")
            
        print("Starting question extraction and rephrasing pipeline...")
        
        # Step 1: Scrape text from PDF/LaTeX
        print("\nStep 1: Scraping textbook content...")
        pdf_text = read_data_from_latex(publication=conf_data['publication'] , chapter_name=conf_data['chapter_name'])
        if not pdf_text:
            raise ValueError("No text content extracted from textbook")
        print(f"Successfully extracted {len(pdf_text)} text segments")
            
        # Step 2: Extract questions from text
        print("\nStep 2: Extracting questions from text...")
        extracted_questions = extract_questions(
            subject=conf_data["subject"],
            grade=conf_data["grade"],
            chapter_name=conf_data["chapter_name"],
            chapter_texts=pdf_text.split("\n")
        )
        
        if not extracted_questions:
            raise ValueError("No questions were extracted from the text")
        print(f"Successfully extracted questions")
            
        # Step 3: Clean and rephrase questions
        print("\nStep 3: Cleaning and rephrasing questions...")
        final_rephrased_questions = rephrase_questions(
            conf_data=conf_data,
            extracted_questions=extracted_questions,
            topics=topics
        )
        if not final_rephrased_questions:
            raise ValueError("No questions were successfully rephrased")
        print("Successfully rephrased questions")
            
        # Return results
        print("\nPipeline completed successfully!")
        return extracted_questions, final_rephrased_questions
        
    except ValueError as val_err:
        print(f"Validation error: {str(val_err)}")
        print(f"Traceback:\n{traceback.format_exc()}")
        return None, None
        
    except Exception as e:
        print(f"Unexpected error in question processing pipeline: {str(e)}")
        print(f"Exception type: {type(e).__name__}")
        print(f"Traceback:\n{traceback.format_exc()}")
        return None, None
        
    finally:
        print("\nQuestion processing pipeline finished")

In [27]:
import pandas as pd

def process_filename(filename):
    """
    Process filename by stripping whitespace and converting to lowercase
    """
    return filename.strip().lower()

# Match subject to determine sheet name
match subject:
    case "mathematics":
        sheet_name = "G10 Maths"
    case "biology":
        sheet_name = "G10 Science"
    case "physics":
        sheet_name = "G10 Science"
    case "chemistry":
        sheet_name = "G10 Science"

# Read Excel file and process the filename
file_name = process_filename("LEAP Course Creation - Topic LU Prerequisite Misconceptions.xlsx")
misconceptions_df = pd.read_excel(file_name, sheet_name=sheet_name)

# Clean column names
misconceptions_df.columns = misconceptions_df.columns.str.strip()
misconceptions_df.fillna("", inplace=True)

# Process chapter name from conf_data for comparison
chapter_name = conf_data["chapter_name"].strip().lower()

# Get unique topics for the specified chapter
tempTopics = list(misconceptions_df[
    misconceptions_df["Chapter (NCERT/AcadAlly's name)"].str.strip().str.lower() == chapter_name
]["Topic"].unique())

# Create topics dictionary with processed IDs
topics = {}
for i in range(len(tempTopics)):
    topic_id = f"{conf_data['grade']}_{conf_data['subject']}_{chapter_name}_{i+1}"
    topics[topic_id] = tempTopics[i].strip()

print("Topics with ID: ", topics, "\n\n", topics.values())

# Process misconceptions and LUs
total_misconceptions = {}
total_LUs = {}

for i, row in misconceptions_df.iterrows():
    topic = str(row["Topic"]).strip()
    
    if topic in topics.values():
        # Process LUs
        if topic in total_LUs:
            total_LUs[topic].append(row["LU text"])
        else:
            total_LUs[topic] = [row["LU text"]]

        # Process misconceptions
        temp_misconceptions = [
            row[f"Misconceptions {i}"] 
            for i in range(1, 11) 
            if row[f"Misconceptions {i}"] != ""
        ]

        if topic in total_misconceptions:
            total_misconceptions[topic].extend(temp_misconceptions)
        else:
            total_misconceptions[topic] = temp_misconceptions

print(total_LUs, total_misconceptions, sep="\n\n-------------------------------------------------------------------\n\n")

Topics with ID:  {'10_chemistry_chemical reactions and equations_1': 'Writing Chemical Reactions and Equations', '10_chemistry_chemical reactions and equations_2': 'Balancing Chemical Equations', '10_chemistry_chemical reactions and equations_3': 'Combination and Decomposition Reactions', '10_chemistry_chemical reactions and equations_4': 'Displacement and Double Displacement Reactions', '10_chemistry_chemical reactions and equations_5': 'Redox Reactions and Their Effects'} 

 dict_values(['Writing Chemical Reactions and Equations', 'Balancing Chemical Equations', 'Combination and Decomposition Reactions', 'Displacement and Double Displacement Reactions', 'Redox Reactions and Their Effects'])
{'Writing Chemical Reactions and Equations': ['Definition and Indicators of Chemical Reactions', 'Writing Chemical Equations'], 'Balancing Chemical Equations': ['Steps to balance a Chemical equation', 'Balancing the chemical equations '], 'Combination and Decomposition Reactions': ['Combination Re

In [28]:
total_LUs

{'Writing Chemical Reactions and Equations': ['Definition and Indicators of Chemical Reactions',
  'Writing Chemical Equations'],
 'Balancing Chemical Equations': ['Steps to balance a Chemical equation',
  'Balancing the chemical equations '],
 'Combination and Decomposition Reactions': ['Combination Reactions ',
  'Exothermic and Endothermic Reactions (Examples and Differences)',
  'Decomposition Reactions '],
 'Displacement and Double Displacement Reactions': ['Displacement Reactions ',
  'Double Displacement Reactions (including precipitation reactions)'],
 'Redox Reactions and Their Effects': ['Redox Reactions ',
  'Effects of Oxidation Reactions']}

In [29]:
# function call for execution
extracted_raw_questions_json ,clean_rephrased_questions_json = extract_rephrase_questions(conf_data, topics)
# clean_rephrased_questions_json

Starting question extraction and rephrasing pipeline...

Step 1: Scraping textbook content...
Successfully processed LaTeX file from zip: C:\Users\ANIKET~1\AppData\Local\Temp\tmplg7txxnc\2025_01_24_65cd09b9c0c686f18039g\2025_01_24_65cd09b9c0c686f18039g.tex
Successfully extracted 34483 text segments

Step 2: Extracting questions from text...

Processing chunk 1/2

Raw response content: {
    "case_study_1": {
        "description": "Chemical equation is a method of representing a chemical reaction with the help of symbols and formulae of the substances involved in it. In a chemical 

Parsed response: {'case_study_1': {'description': 'Chemical equation is a method of representing a chemical reaction with the help of symbols and formulae of the substances involved in it. In a chemical equation, the substances which combine or react are called reactants and new substances produced are called products. A chemical equation is a short hand method of representing a chemical reaction. A balance

In [30]:
print("Chapter name: ",chapter, ",  Raw questions: ",  len(extracted_raw_questions_json))
print("Chapter name: ",chapter, ",  Cleaned questions: ",  len(clean_rephrased_questions_json))

Chapter name:  Chemical Reactions and Equations ,  Raw questions:  11
Chapter name:  Chemical Reactions and Equations ,  Cleaned questions:  10


In [33]:
def convert_case_studies_to_df(case_studies_data: dict) -> pd.DataFrame:
   try:
       rows = []
       
       for case_study_key, content in case_studies_data.items():
           # Get case study level data
           description = content.get("description", "")
           topic = content.get("topic", "")
           topic_id = content.get("topic_id", "")
           
           # Process each question in the case study
           for question in content.get("questions", []):
               row = {
                   "case_study": case_study_key,
                   "description": description,
                   "topic": topic, 
                   "topic_id": topic_id,
                   "question": question.get("question", ""),
                   "hint": question.get("hint", ""),
                   "solution": question.get("solution", ""),
                   "correct_option": question.get("correct_option", ""),
                   "option1": question.get("option1", ""),
                   "dr1": question.get("dr1", ""),
                   "option2": question.get("option2", ""),
                   "dr2": question.get("dr2", ""),
                   "option3": question.get("option3", ""),
                   "dr3": question.get("dr3", "")
               }
               rows.append(row)
       
       # Create DataFrame
       df = pd.DataFrame(rows)
       
       # Reorder columns 
       column_order = [
           "case_study", "description", "topic", "topic_id", 
           "question", "hint", "solution", "correct_option",
           "option1", "dr1", "option2", "dr2", "option3", "dr3"
       ]
       df = df[column_order]
       
       print(f"Created DataFrame with {len(df)} questions from {len(case_studies_data)} case studies")
       return df
       
   except Exception as e:
       print(f"Error converting to DataFrame: {str(e)}")
       return pd.DataFrame()

In [34]:
clean_rephrased_questions_json['case_study_1']

{'description': '\\begin{aligned} & \\text{Chemical equation is a method of representing a chemical reaction with the help of symbols and formulae of the substances involved in it.} \\\\ & \\text{In a chemical equation, the substances which combine or react are called reactants and new substances produced are called products.} \\\\ & \\text{A chemical equation is a short hand method of representing a chemical reaction.} \\\\ & \\text{A balanced chemical equation has equal number of atoms of different elements in the reactants and products side.} \\\\ & \\text{An unbalanced chemical equation has unequal number of atoms of one or more elements in reactants and products.} \\\\ & \\text{Formulae of elements and compounds are not changed to balance an equation.} \\\\ \\end{aligned}',
 'questions': [{'question': '\\begin{aligned} & \\text{Consider the following reaction:} \\\\ & \\quad p \\mathrm{Mg}_{3} \\mathrm{~N}_{2}+q \\mathrm{H}_{2} \\mathrm{O} \\rightarrow r \\mathrm{Mg}(\\mathrm{OH})

# Solution and Distractors(Based on misconceptions)

In [35]:
llm = ChatOpenAI(model="gpt-4o", temperature=0.2, api_key = OPENAI_API_KEY ,  request_timeout=30.0)

In [36]:
questions_df = convert_case_studies_to_df(clean_rephrased_questions_json)
questions_df

Created DataFrame with 50 questions from 10 case studies


,case_study,description,topic,topic_id,question,hint,solution,correct_option,option1,dr1,option2,dr2,option3,dr3
0,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{Consider the following...,,,,,,,,,
1,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{Which of the following...,,,,,,,,,
2,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{The balancing of chemi...,,,,,,,,,
3,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{Which of the following...,,,,,,,,,
4,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{Which of the following...,,,,,,,,,
5,case_study_2,\begin{aligned} & \text{In decomposition react...,Combination and Decomposition Reactions,10_chemistry_chemical reactions and equations_3,\begin{aligned} & \text{Which of the following...,,,,,,,,,
6,case_study_2,\begin{aligned} & \text{In decomposition react...,Combination and Decomposition Reactions,10_chemistry_chemical reactions and equations_3,\begin{aligned} & \text{In the reaction } 2 \m...,,,,,,,,,
7,case_study_2,\begin{aligned} & \text{In decomposition react...,Combination and Decomposition Reactions,10_chemistry_chemical reactions and equations_3,\begin{aligned} & \text{Amino acid is formed b...,,,,,,,,,
8,case_study_2,\begin{aligned} & \text{In decomposition react...,Combination and Decomposition Reactions,10_chemistry_chemical reactions and equations_3,\begin{aligned} & \text{Silver chloride on exp...,,,,,,,,,
9,case_study_2,\begin{aligned} & \text{In decomposition react...,Combination and Decomposition Reactions,10_chemistry_chemical reactions and equations_3,\begin{aligned} & \text{What type of chemical ...,,,,,,,,,


In [37]:
import json
def parse_Response(response):
    if isinstance(response, str):
        # print(response)
        start_index = response.find("{")
        end_index = response.rfind("}")
        if start_index != -1 and end_index != -1:
            valid_json_content = response[start_index : end_index + 1]
            try:
                JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
                # append_list_to_file(JSON_response)
                return JSON_response
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON response: {e.__class__.__name__} - {e}\n\n Still trying to work on particular exceptions ...")
                if str(e).startswith("Extra data"):
                    json_parts = valid_json_content.split('}\n{')
                    if json_parts:
                        first_json = json_parts[0] + '}'
                        return parse_Response(first_json)  
                elif str(e).startswith("Invalid \escape"):
                    print("Before removing \\ issue: ", valid_json_content)
                    str1 = valid_json_content.replace("\\\\\\\\\\", "fvback")
                    str1 = str1.replace("\\\\\\\\", "frback")
                    str1 = str1.replace("\\\\\\", "trlback")
                    str1 = str1.replace("\\\\", "dblback")
                    str1 = str1.replace("\\", "\\\\")
                    str1 = str1.replace("fvback","\\\\\\\\\\")
                    str1 = str1.replace("frback", "\\\\\\\\")
                    str1 = str1.replace("trlback","\\\\\\")
                    strfinal = str1.replace("dblback","\\\\")
                    print("After removing \\ issue: ", strfinal)
                    JSON_response = json.loads(strfinal.replace("\n", ""))
                    return JSON_response
                else:
                    print("Actual Content: ", valid_json_content)
        else:
            print("No valid JSON content found in the response.")
        # time.sleep(50)
    elif isinstance(response, dict):
        return response
    else:
        print("No response message found", type(response))

<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_25956\3339088456.py:10: SyntaxWarning: invalid escape sequence '\('
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_25956\3339088456.py:10: SyntaxWarning: invalid escape sequence '\)'
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_25956\3339088456.py:20: SyntaxWarning: invalid escape sequence '\e'
  elif str(e).startswith("Invalid \escape"):


In [43]:
data = clean_rephrased_questions_json

In [44]:
clean_rephrased_questions_json = dict(list(clean_rephrased_questions_json.items())[:2])

In [45]:
## Generating solutions
prompt_template = PromptTemplate(
    input_variables=["system_prompt", "question", "topic", "topic_id", "lulist"],
    template="{system_prompt}\n\n\nGenerate a hint, and solution for the given question. Also rephrase the given question according to the specified steps.\nHere are the required context:\nQuestion: {question}\nLearning unit data which defines scope from which solution should be generated: {lulist}\nTopic: {topic}\nTopic ID: {topic_id}\n\nReturn the output in the specified JSON format."
)

chain = LLMChain(llm=llm, prompt=prompt_template)
# chain = prompt_template | llm
# Function to process each question and generate hint, solution, and final answer
def generate_explanation(subject, question, topic, topic_id):
    try:
        match subject:
            case "mathematics":
                system = MATHEMATICS_SOLUTION_SYSTEM_PROMPT
            case "biology":
                system = BIOLOGY_SOLUTION_SYSTEM_PROMPT
            case "physics":
                system = PHYSICS_SOLUTION_SYSTEM_PROMPT
            case "chemistry":
                system = CHEMISTRY_SOLUTION_SYSTEM_PROMPT

        response = chain.run(
            system_prompt=system,
            lulist=total_LUs,
            question=question,
            topic=topic,
            topic_id=topic_id
        )
        return response
    except Exception as e:
        print(f"Error processing question: {question}\nError: {str(e)}")
        return None
# parser = PydanticOutputParser(pydantic_object=sol_data)
# global explanations

def loopForSolution(subject, data):
   try:
       for case_study_key, content in data.items():
           print(f"\nProcessing solutions for {case_study_key}")
           retries = 0
           
           # Process each question in case study
           for i, question in enumerate(content["questions"]):
               while retries < 3:  # Try up to 3 times for each question
                   try:
                       print(f"Processing question {i + 1} out of {len(content['questions'])}")
                       explanation = generate_explanation(
                           subject=subject,
                           question=question["question"],
                           topic=content["topic"],
                           topic_id=content["topic_id"]
                       )
                       
                       parsed_explanation = parse_Response(explanation)
                       if parsed_explanation:
                           # Update question with solution data
                           question.update({
                               "hint": parsed_explanation.get("hint", ""),
                               "solution": parsed_explanation.get("solution", "")
                           })
                           retries = 0
                           break
                       
                       else:
                           print(f"Failed to parse solution for question {i + 1}, attempt {retries + 1}/3")
                           retries += 1
                           
                   except Exception as e:
                       print(f"Error processing question {i + 1}: {str(e)}")
                       retries += 1
               
               if retries == 3:
                   print(f"Failed to process question {i + 1} after 3 attempts, moving to next question")
                   retries = 0  # Reset for next question
           
       return data

   except Exception as e:
       print(f"Error in loopForSolution: {str(e)}")
       return data
   
loopForSolution(subject, clean_rephrased_questions_json)

C:\Users\aniket singh\AppData\Local\Temp\ipykernel_25956\4087258864.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt_template)
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_25956\4087258864.py:22: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = chain.run(



Processing solutions for case_study_1
Processing question 1 out of 5
Processing question 2 out of 5
Processing question 3 out of 5
Processing question 4 out of 5
Processing question 5 out of 5

Processing solutions for case_study_2
Processing question 1 out of 5
Processing question 2 out of 5
Processing question 3 out of 5
Processing question 4 out of 5
Processing question 5 out of 5


{'case_study_1': {'description': '\\begin{aligned} & \\text{Chemical equation is a method of representing a chemical reaction with the help of symbols and formulae of the substances involved in it.} \\\\ & \\text{In a chemical equation, the substances which combine or react are called reactants and new substances produced are called products.} \\\\ & \\text{A chemical equation is a short hand method of representing a chemical reaction.} \\\\ & \\text{A balanced chemical equation has equal number of atoms of different elements in the reactants and products side.} \\\\ & \\text{An unbalanced chemical equation has unequal number of atoms of one or more elements in reactants and products.} \\\\ & \\text{Formulae of elements and compounds are not changed to balance an equation.} \\\\ \\end{aligned}',
  'questions': [{'question': '\\begin{aligned} & \\text{Consider the following reaction:} \\\\ & \\quad p \\mathrm{Mg}_{3} \\mathrm{~N}_{2}+q \\mathrm{H}_{2} \\mathrm{O} \\rightarrow r \\mathrm

### Generating Distractors

In [46]:
data= convert_case_studies_to_df(clean_rephrased_questions_json)
data

Created DataFrame with 10 questions from 2 case studies


,case_study,description,topic,topic_id,question,hint,solution,correct_option,option1,dr1,option2,dr2,option3,dr3
0,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{Consider the following...,\begin{aligned} &\text{To balance the chemical...,\begin{aligned} &\text{The given chemical reac...,,,,,,,
1,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{Which of the following...,\begin{aligned} & \text{Consider what a balanc...,\begin{aligned} & \text{A balanced chemical eq...,,,,,,,
2,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{The balancing of chemi...,\begin{aligned} &\text{Think about the fundame...,\begin{aligned} &\text{The balancing of chemic...,,,,,,,
3,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{Which of the following...,\begin{aligned} &\text{To determine if a chemi...,\begin{aligned} &\text{A chemical equation is ...,,,,,,,
4,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{Which of the following...,\begin{aligned} &\text{Consider the law of con...,\begin{aligned} &\text{A balanced chemical equ...,,,,,,,
5,case_study_2,\begin{aligned} & \text{In decomposition react...,Combination and Decomposition Reactions,10_chemistry_chemical reactions and equations_3,\begin{aligned} & \text{Which of the following...,\begin{aligned} &\text{A decomposition reactio...,\begin{aligned} &\text{In a decomposition reac...,,,,,,,
6,case_study_2,\begin{aligned} & \text{In decomposition react...,Combination and Decomposition Reactions,10_chemistry_chemical reactions and equations_3,\begin{aligned} & \text{In the reaction } 2 \m...,\begin{aligned} &\text{Identify the type of re...,\begin{aligned} &\text{The given reaction is a...,,,,,,,
7,case_study_2,\begin{aligned} & \text{In decomposition react...,Combination and Decomposition Reactions,10_chemistry_chemical reactions and equations_3,\begin{aligned} & \text{Amino acid is formed b...,\begin{aligned} &\text{Consider which macromol...,\begin{aligned} &\text{Amino acids are formed ...,,,,,,,
8,case_study_2,\begin{aligned} & \text{In decomposition react...,Combination and Decomposition Reactions,10_chemistry_chemical reactions and equations_3,\begin{aligned} & \text{Silver chloride on exp...,\begin{aligned} &\text{Consider the type of re...,\begin{aligned} &\text{When silver chloride (A...,,,,,,,
9,case_study_2,\begin{aligned} & \text{In decomposition react...,Combination and Decomposition Reactions,10_chemistry_chemical reactions and equations_3,\begin{aligned} & \text{What type of chemical ...,\begin{aligned} &\text{Consider the process of...,\begin{aligned} &\text{When electricity is pas...,,,,,,,


In [47]:
clean_rephrased_questions_json['case_study_1']

{'description': '\\begin{aligned} & \\text{Chemical equation is a method of representing a chemical reaction with the help of symbols and formulae of the substances involved in it.} \\\\ & \\text{In a chemical equation, the substances which combine or react are called reactants and new substances produced are called products.} \\\\ & \\text{A chemical equation is a short hand method of representing a chemical reaction.} \\\\ & \\text{A balanced chemical equation has equal number of atoms of different elements in the reactants and products side.} \\\\ & \\text{An unbalanced chemical equation has unequal number of atoms of one or more elements in reactants and products.} \\\\ & \\text{Formulae of elements and compounds are not changed to balance an equation.} \\\\ \\end{aligned}',
 'questions': [{'question': '\\begin{aligned} & \\text{Consider the following reaction:} \\\\ & \\quad p \\mathrm{Mg}_{3} \\mathrm{~N}_{2}+q \\mathrm{H}_{2} \\mathrm{O} \\rightarrow r \\mathrm{Mg}(\\mathrm{OH})

In [48]:
# Define the prompt template for generating misconceptions
prompt_template = PromptTemplate(
    input_variables=["system_prompt","topic", "topic_id", "question", "hint", "solution"],
    template="{system_prompt}\n\nGenerate one correct option and appropriate incorrect options in latex using given solution, hint, and misconceptions for the given question.\n\nTopic ID: {topic_id}\nTopic: {topic}\nQuestion: {question}\nHint: {hint}\nSolution: {solution}\n\nReturn the output in the specified JSON format."
)

# Extract relevant information from the 'explanation' column

def extract_explanation_details(explanation):
    try:
        explanation_data = json.loads(explanation)
        hint = explanation_data.get('hint', 'No hint available')
        solution = explanation_data.get('solution', 'No solution available')
        return hint, solution
    except json.JSONDecodeError:
        return 'No hint available', 'No solution available'

# Function to generate misconceptions-based incorrect options

def generate_incorrect_options(row):

    chain = LLMChain(llm=llm, prompt=prompt_template)
    try:
        response = chain.run(
            system_prompt=DISTRACTOR_SYSTEM_PROMPT,
            topic=row['topic'],
            topic_id=row['topic_id'],
            question=row['question'],
            hint=row['hint'],
            solution=row['solution']
        )
        # if response.strip().startswith("```json"):
        #     response = response.strip().strip("```json").strip("```").strip()
        return response
    except Exception as e:
        print(f"Unexpected error: {str(e)}")
        return None

def loopForDistractor(case_studies_data: dict) -> dict:
   try:
       for case_study_key, content in case_studies_data.items():
           print(f"\nProcessing distractors for {case_study_key}")
           
           for i, question in enumerate(content.get("questions", [])):
               retries = 0
               while retries < 3:  
                   try:
                       print(f"Processing question {i + 1} out of {len(content['questions'])}")
                       
                       misconception_options = generate_incorrect_options({
                           'question': question.get("question", ""),
                           'solution': question.get("solution", ""),
                           'hint': question.get("hint", ""),
                           'topic': content.get("topic", ""),
                           'topic_id': content.get("topic_id", "")
                       })
                       
                       parsed_options = parse_Response(misconception_options)
                       if parsed_options:
                           question.update({
                               "correct_option": parsed_options.get("correct_option", ""),
                               "option1": parsed_options.get("option1", {}).get("option", ""),
                               "dr1": parsed_options.get("option1", {}).get("rationale", ""),
                               "option2": parsed_options.get("option2", {}).get("option", ""),
                               "dr2": parsed_options.get("option2", {}).get("rationale", ""),
                               "option3": parsed_options.get("option3", {}).get("option", ""),
                               "dr3": parsed_options.get("option3", {}).get("rationale", "")
                           })
                           retries = 0  
                           break
                       else:
                           print(f"Failed to parse options for question {i + 1}, attempt {retries + 1}/3")
                           retries += 1
                           
                   except Exception as e:
                       print(f"Error processing question {i + 1}: {str(e)}")
                       retries += 1
               
               if retries == 3:
                   print(f"Failed to process question {i + 1} after 3 attempts, moving to next question")
                   question.update({
                       "correct_option": "",
                       "option1": "", "dr1": "",
                       "option2": "", "dr2": "",
                       "option3": "", "dr3": ""
                   })
           
       return case_studies_data

   except Exception as e:
       print(f"Error in loopForDistractor: {str(e)}")
       return case_studies_data
loopForDistractor(clean_rephrased_questions_json)
# data['misconception_options'] = misconception_options


Processing distractors for case_study_1
Processing question 1 out of 5
Processing question 2 out of 5
Processing question 3 out of 5
Processing question 4 out of 5
Processing question 5 out of 5

Processing distractors for case_study_2
Processing question 1 out of 5
Processing question 2 out of 5
Processing question 3 out of 5
Processing question 4 out of 5
Processing question 5 out of 5


{'case_study_1': {'description': '\\begin{aligned} & \\text{Chemical equation is a method of representing a chemical reaction with the help of symbols and formulae of the substances involved in it.} \\\\ & \\text{In a chemical equation, the substances which combine or react are called reactants and new substances produced are called products.} \\\\ & \\text{A chemical equation is a short hand method of representing a chemical reaction.} \\\\ & \\text{A balanced chemical equation has equal number of atoms of different elements in the reactants and products side.} \\\\ & \\text{An unbalanced chemical equation has unequal number of atoms of one or more elements in reactants and products.} \\\\ & \\text{Formulae of elements and compounds are not changed to balance an equation.} \\\\ \\end{aligned}',
  'questions': [{'question': '\\begin{aligned} & \\text{Consider the following reaction:} \\\\ & \\quad p \\mathrm{Mg}_{3} \\mathrm{~N}_{2}+q \\mathrm{H}_{2} \\mathrm{O} \\rightarrow r \\mathrm

In [49]:
data = convert_case_studies_to_df(clean_rephrased_questions_json)
data

Created DataFrame with 10 questions from 2 case studies


,case_study,description,topic,topic_id,question,hint,solution,correct_option,option1,dr1,option2,dr2,option3,dr3
0,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{Consider the following...,\begin{aligned} &\text{To balance the chemical...,\begin{aligned} &\text{The given chemical reac...,"\begin{aligned} &\quad \text{1, 6, 3, 2} \end{...","\begin{aligned} &\quad \text{1, 3, 3, 2} \end{...",\begin{aligned} &\quad \text{Confused the numb...,"\begin{aligned} &\quad \text{1, 6, 2, 2} \end{...",\begin{aligned} &\quad \text{Forgot to balance...,"\begin{aligned} &\quad \text{1, 6, 3, 1} \end{...",\begin{aligned} &\quad \text{Ignored the corre...
1,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{Which of the following...,\begin{aligned} & \text{Consider what a balanc...,\begin{aligned} & \text{A balanced chemical eq...,\begin{aligned} & \text{Reaction rate or condi...,\begin{aligned} & \text{Conservation of mass} ...,\begin{aligned} & \text{Confused the concept o...,\begin{aligned} & \text{Identities of reactant...,\begin{aligned} & \text{Forgot that a balanced...,\begin{aligned} & \text{Molecular ratios} \end...,\begin{aligned} & \text{Ignored that balanced ...
2,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{The balancing of chemi...,\begin{aligned} &\text{Think about the fundame...,\begin{aligned} &\text{The balancing of chemic...,\begin{aligned} &\text{Law of Conservation of ...,\begin{aligned} &\text{Law of Definite Proport...,\begin{aligned} &\text{Confused the concept wi...,\begin{aligned} &\text{Law of Multiple Proport...,\begin{aligned} &\text{Misunderstood the law t...,\begin{aligned} &\text{Law of Conservation of ...,\begin{aligned} &\text{Ignored the distinction...
3,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{Which of the following...,\begin{aligned} &\text{To determine if a chemi...,\begin{aligned} &\text{A chemical equation is ...,\begin{aligned} &\text{H}_2 + \text{O}_2 \righ...,\begin{aligned} &\text{2H}_2 + \text{O}_2 \rig...,\begin{aligned} &\text{Confused the balanced e...,\begin{aligned} &\text{C}_2\text{H}_4 + 3\text...,\begin{aligned} &\text{Forgot that this equati...,\begin{aligned} &\text{N}_2 + 3\text{H}_2 \rig...,\begin{aligned} &\text{Ignored the fact that t...
4,case_study_1,\begin{aligned} & \text{Chemical equation is a...,Balancing Chemical Equations,10_chemistry_chemical reactions and equations_2,\begin{aligned} & \text{Which of the following...,\begin{aligned} &\text{Consider the law of con...,\begin{aligned} &\text{A balanced chemical equ...,\begin{aligned} &\text{A balanced chemical equ...,\begin{aligned} &\text{A balanced chemical equ...,\begin{aligned} &\text{Confused the concept of...,\begin{aligned} &\text{A balanced chemical equ...,\begin{aligned} &\text{Forgot that the number ...,\begin{aligned} &\text{A balanced chemical equ...,\begin{aligned} &\text{Ignored the requirement...
5,case_study_2,\begin{aligned} & \text{In decomposition react...,Combination and Decomposition Reactions,10_chemistry_chemical reactions and equations_3,\begin{aligned} & \text{Which of the following...,\begin{aligned} &\text{A decomposition reactio...,\begin{aligned} &\text{In a decomposition reac...,\begin{aligned} &\text{CaCO}_3 \rightarrow \te...,\begin{aligned} &\text{2H}_2 + \text{O}_2 \rig...,\begin{aligned} &\text{Confused the combinatio...,\begin{aligned} &\text{N}_2 + \text{3H}_2 \rig...,\begin{aligned} &\text{Forgot that this is a s...,\begin{aligned} &\text{C}_6\text{H}_{12}\text{...,\begin{aligned} &\text{Ignored that this is a ...
6,case_study_2,\

### Output Formatting & Pushing into the DB

In [50]:
def process_case_studies(data, conf_data):
   try:
       # Common metadata for all case studies
       common_metadata = {
           "grade": conf_data['grade'],
           "board": "CBSE",
           "subject": subject,
           "chapter_name": conf_data['chapter_name'],
           "publication": conf_data['publication']
       }

       # Group by case study and process rows
       case_study_data = {}
       for idx, row in data.iterrows():
           case_key = row["case_study"]
           
           # Create question data
           question_data = {
               "topic_id": row["topic_id"],
               "topic_name": row["topic"],
               "question": [{"content": row["question"]}],
               "hint": [{"content": row["hint"]}],
               "solution": [{"content": row["solution"]}],
               "final_answer": [{"content": row["correct_option"]}],
               "option1": [{"content": row["option1"]}],
               "dr1": [{"content": row["dr1"]}],
               "option2": [{"content": row["option2"]}],
               "dr2": [{"content": row["dr2"]}],
               "option3": [{"content": row["option3"]}],
               "dr3": [{"content": row["dr3"]}]
           }
           
           # Add to case study group
           if case_key not in case_study_data:
               case_study_data[case_key] = {
                   **common_metadata,  # Add common metadata
                   "case_study": row["description"],
                   "questions": []
               }
           case_study_data[case_key]["questions"].append(question_data)

       final_output = list(case_study_data.values())
       return final_output

   except Exception as e:
       print(f"Error processing case studies: {str(e)}")
       return []

# Process the data
final_output = process_case_studies(data, conf_data)
final_output

[{'grade': '10',
  'board': 'CBSE',
  'subject': 'chemistry',
  'chapter_name': 'Chemical Reactions and Equations',
  'publication': 'mtg',
  'case_study': '\\begin{aligned} & \\text{Chemical equation is a method of representing a chemical reaction with the help of symbols and formulae of the substances involved in it.} \\\\ & \\text{In a chemical equation, the substances which combine or react are called reactants and new substances produced are called products.} \\\\ & \\text{A chemical equation is a short hand method of representing a chemical reaction.} \\\\ & \\text{A balanced chemical equation has equal number of atoms of different elements in the reactants and products side.} \\\\ & \\text{An unbalanced chemical equation has unequal number of atoms of one or more elements in reactants and products.} \\\\ & \\text{Formulae of elements and compounds are not changed to balance an equation.} \\\\ \\end{aligned}',
  'questions': [{'topic_id': '10_chemistry_chemical reactions and equa

In [51]:
final_output[0]

{'grade': '10',
 'board': 'CBSE',
 'subject': 'chemistry',
 'chapter_name': 'Chemical Reactions and Equations',
 'publication': 'mtg',
 'case_study': '\\begin{aligned} & \\text{Chemical equation is a method of representing a chemical reaction with the help of symbols and formulae of the substances involved in it.} \\\\ & \\text{In a chemical equation, the substances which combine or react are called reactants and new substances produced are called products.} \\\\ & \\text{A chemical equation is a short hand method of representing a chemical reaction.} \\\\ & \\text{A balanced chemical equation has equal number of atoms of different elements in the reactants and products side.} \\\\ & \\text{An unbalanced chemical equation has unequal number of atoms of one or more elements in reactants and products.} \\\\ & \\text{Formulae of elements and compounds are not changed to balance an equation.} \\\\ \\end{aligned}',
 'questions': [{'topic_id': '10_chemistry_chemical reactions and equations_2

#### Latex Formatting to store in DB

In [52]:
from config import MONGO_URL , DB_NAME

In [56]:
dbCollection = "Latex_CaseStudy"
client = AsyncIOMotorClient(MONGO_URL)

In [57]:
async def pushToDB(dbContent, subject):
    try:
        db = client[DB_NAME]
        que_collection = db[dbCollection]

        dbContent["status"] = "not_reviewed"
        dbContent["comment"] = ""
        dbContent["testFlag"] = "true"
        dbContent["subject"] = subject
        await que_collection.insert_one(dbContent)
        print("Data Uploaded to the database successfully.")
    except Exception as e:
        print("Exception Occurred: ", type(e).__name__, "–", e, "\n", traceback.format_exc())

In [58]:
for doc in final_output:
    await pushToDB(doc, subject)
client.close()

Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
